In [13]:
# ============================================================
# GROUPDNA - WHATSAPP GROUP CHAT ANALYZER
# ============================================================
# Concepts Used:
# Strings, Lists, Dictionaries, Sets, Tuples, Loops,
# Conditionals, Functions, NumPy, File I/O
#
# AI Assistance: Used as a learning aid. Final code structure
# and understanding should be reviewed by the student.
# ============================================================

import numpy as np
from datetime import datetime, timedelta


# ============================================================
# SETTINGS
# ============================================================

FILE_NAME = "/content/hostel_bois.txt"


# ============================================================
# FEATURE 1: CHAT PARSER
# ============================================================

def parse_chat(file_name):

    messages = []
    participants = set()

    system_messages = 0
    deleted_messages = 0
    media_messages = 0

    with open(file_name, "r", encoding="utf-8") as file:

        for line in file:

            line = line.strip()

            if line == "":
                continue

            # ------------------------------------------------
            # Check for WhatsApp date/time format
            # Example:
            # 01/04/24, 10:30 pm - Rahul: Hello
            # ------------------------------------------------

            if " - " not in line:
                system_messages += 1
                continue

            try:
                date_time_part, message_part = line.split(" - ", 1)

                if ", " not in date_time_part:
                    system_messages += 1
                    continue

                date_part, time_part = date_time_part.split(", ", 1)

                timestamp = datetime.strptime(
                    date_part + ", " + time_part,
                    "%d/%m/%y, %I:%M %p"
                )

            except:
                system_messages += 1
                continue

            # ------------------------------------------------
            # Separate sender and message
            # ------------------------------------------------

            if ": " not in message_part:
                system_messages += 1
                continue

            sender, message_text = message_part.split(": ", 1)

            sender = sender.strip()
            message_text = message_text.strip()

            # ------------------------------------------------
            # Deleted message
            # ------------------------------------------------

            if "This message was deleted" in message_text:
                deleted_messages += 1
                continue

            # ------------------------------------------------
            # Media message
            # ------------------------------------------------

            if "<Media omitted>" in message_text:
                media_messages += 1

            # ------------------------------------------------
            # Tuple usage
            # ------------------------------------------------

            message_tuple = (
                timestamp,
                sender,
                message_text
            )

            # ------------------------------------------------
            # Dictionary usage
            # ------------------------------------------------

            message_record = {
                "timestamp": message_tuple[0],
                "sender": message_tuple[1],
                "message": message_tuple[2]
            }

            messages.append(message_record)
            participants.add(sender)

    return (
        messages,
        participants,
        system_messages,
        deleted_messages,
        media_messages
    )


# ============================================================
# FEATURE 2: GROUP OVERVIEW
# ============================================================

def group_overview(messages, participants):

    if len(messages) == 0:
        print("No messages found.")
        return

    dates = []

    for msg in messages:
        dates.append(msg["timestamp"].date())

    start_date = min(dates)
    end_date = max(dates)

    total_days = (end_date - start_date).days + 1

    person_counts = {}

    for person in participants:
        person_counts[person] = 0

    for msg in messages:
        sender = msg["sender"]

        if sender not in person_counts:
            person_counts[sender] = 0

        person_counts[sender] += 1

    print("\n" + "=" * 65)
    print("GROUP OVERVIEW")
    print("=" * 65)

    print("Total Messages :", len(messages))
    print("Start Date     :", start_date)
    print("End Date       :", end_date)
    print("Total Days     :", total_days)
    print("Participants   :", len(participants))

    print("\nMessages by Participant")
    print("-" * 65)

    sorted_people = sorted(
        person_counts.items(),
        key=lambda item: item[1],
        reverse=True
    )

    for person, count in sorted_people:
        print(f"{person:<15} : {count}")


# ============================================================
# FEATURE 3: MOST ACTIVE DAY AND HOUR
# ============================================================

def busiest_day_and_hour(messages):

    # Add a check for empty messages list
    if len(messages) == 0:
        print("\n" + "=" * 65)
        print("MOST ACTIVE DAY AND HOUR")
        print("=" * 65)
        print("No messages available to determine busiest day and hour.")
        return

    day_counts = {}
    hour_counts = {}

    for msg in messages:

        date_value = msg["timestamp"].date()
        hour_value = msg["timestamp"].hour

        if date_value not in day_counts:
            day_counts[date_value] = 0

        if hour_value not in hour_counts:
            hour_counts[hour_value] = 0

        day_counts[date_value] += 1
        hour_counts[hour_value] += 1

    busiest_day = max(
        day_counts,
        key=day_counts.get
    )

    busiest_hour = max(
        hour_counts,
        key=hour_counts.get
    )

    print("\n" + "=" * 65)
    print("MOST ACTIVE DAY AND HOUR")
    print("=" * 65)

    print(
        f"Most Active Day  : {busiest_day}"
        f" ({day_counts[busiest_day]} messages)"
    )

    print(
        f"Most Active Hour : {busiest_hour:02d}:00"
        f" ({hour_counts[busiest_hour]} messages)"
    )

    return busiest_day, busiest_hour


# ============================================================
# FEATURE 4: NUMPY ACTIVITY HEATMAP
# ============================================================

def activity_heatmap(messages, participants):
    if not messages or not participants:
        print("\n" + "=" * 65)
        print("ACTIVITY HEATMAP")
        print("=" * 65)
        print("No messages or participants available to generate activity heatmap.")
        print("-" * 90)
        return None

    people = sorted(participants)

    # 6 x 24 matrix
    activity = np.zeros(
        (len(people), 24),
        dtype=int
    )

    person_index = {}

    for index, person in enumerate(people):
        person_index[person] = index

    for msg in messages:

        sender = msg["sender"]
        hour = msg["timestamp"].hour

        if sender in person_index:
            activity[
                person_index[sender],
                hour
            ] += 1

    print("\n" + "=" * 65)
    print("ACTIVITY HEATMAP")
    print("=" * 65)

    print("Person".ljust(15), end="")

    for hour in range(24):
        print(f"{hour:02d} ", end="")

    print()

    print("-" * 90)

    maximum = activity.max()

    if maximum == 0:
        maximum = 1

    for i, person in enumerate(people):

        print(person.ljust(15), end="")

        for hour in range(24):

            value = activity[i][hour]

            ratio = value / maximum

            if value == 0:
                symbol = "."
            elif ratio <= 0.25:
                symbol = "░"
            elif ratio <= 0.50:
                symbol = "▒"
            else:
                symbol = "█"

            print(f"{symbol}  ", end="")

        print()

    return activity


# ============================================================
# FEATURE 5: TOP WORDS
# ============================================================

def top_words(messages):

    stop_words = {
        "the", "is", "a", "an", "and", "or",
        "to", "of", "in", "on", "for",
        "with", "i", "you", "me", "my",
        "we", "it", "this", "that", "are",
        "was", "be", "am", "will", "have",
        "has", "had", "he", "she", "they",
        "but", "so", "if", "at", "from",
        "as", "not", "just", "do", "dont",
        "your", "our", "u", "ur"
    }

    word_counts = {}

    punctuation = [
        ".", ",", "!", "?", ":", ";",
        "(", ")", "[", "]", "{", "}",
        "\"", "'", "-", "_", "/", "\\"
    ]

    for msg in messages:

        text = msg["message"].lower()

        for symbol in punctuation:
            text = text.replace(symbol, " ")

        words = text.split()

        for word in words:

            if word in stop_words:
                continue

            if len(word) <= 1:
                continue

            if word not in word_counts:
                word_counts[word] = 0

            word_counts[word] += 1

    sorted_words = sorted(
        word_counts.items(),
        key=lambda item: item[1],
        reverse=True
    )

    print("\n" + "=" * 65)
    print("TOP 10 COMMON WORDS")
    print("=" * 65)

    for i, (word, count) in enumerate(
        sorted_words[:10],
        start=1
    ):
        print(f"{i:2}. {word:<20} : {count}")

    return sorted_words


# ============================================================
# FEATURE 6A: RESPONSE SPEED
# ============================================================

def response_speed(messages):

    if len(messages) < 2:
        return

    response_times = {}

    previous_message = messages[0]

    for current_message in messages[1:]:

        previous_sender = previous_message["sender"]
        current_sender = current_message["sender"]

        if previous_sender != current_sender:

            time_difference = (
                current_message["timestamp"]
                - previous_message["timestamp"]
            ).total_seconds() / 60

            # Ignore extremely large gaps
            if 0 <= time_difference <= 1440:

                if current_sender not in response_times:
                    response_times[current_sender] = []

                response_times[current_sender].append(
                    time_difference
                )

        previous_message = current_message

    print("\n" + "=" * 65)
    print("AVERAGE RESPONSE TIME")
    print("=" * 65)

    for person in sorted(response_times):

        times = response_times[person]

        if len(times) > 0:
            average = sum(times) / len(times)

            print(
                f"{person:<15} : "
                f"{average:.2f} minutes"
            )


# ============================================================
# FEATURE 6B: SILENT STREAKS
# ============================================================

def silent_streaks(messages, participants):

    if len(messages) == 0:
        return

    first_date = min(
        msg["timestamp"].date()
        for msg in messages
    )

    last_date = max(
        msg["timestamp"].date()
        for msg in messages
    )

    all_dates = []

    current_date = first_date

    while current_date <= last_date:

        all_dates.append(current_date)

        current_date += timedelta(days=1)

    active_dates = {}

    for person in participants:
        active_dates[person] = set()

    for msg in messages:

        person = msg["sender"]
        date_value = msg["timestamp"].date()

        if person in active_dates:
            active_dates[person].add(date_value)

    print("\n" + "=" * 65)
    print("SILENT STREAK ANALYSIS")
    print("=" * 65)

    for person in sorted(participants):

        longest_streak = 0
        current_streak = 0

        for date_value in all_dates:

            if date_value not in active_dates[person]:

                current_streak += 1

                if current_streak > longest_streak:
                    longest_streak = current_streak

            else:
                current_streak = 0

        active_count = len(active_dates[person])
        silent_days = len(all_dates) - active_count

        print(
            f"{person:<15} | "
            f"Active: {active_count:2} days | "
            f"Silent: {silent_days:2} days | "
            f"Longest Streak: {longest_streak:2} days"
        )


# ============================================================
# FEATURE 7: PERSONALITY ARCHETYPES
# ============================================================

def calculate_archetypes(messages, participants):

    data = {}

    for person in participants:

        person_messages = []

        for msg in messages:

            if msg["sender"] == person:
                person_messages.append(msg)

        data[person] = {
            "messages": person_messages,
            "count": len(person_messages)
        }

    # --------------------------------------------------------
    # Archetype order is the tie-break rule.
    # --------------------------------------------------------

    archetype_order = [
        "SPAMMER",
        "GROUP MOM",
        "NIGHT OWL",
        "STORYTELLER",
        "DRAMA QUEEN",
        "GHOST",
        "COMEDIAN",
        "QUESTION MASTER"
    ]

    results = {}

    # ========================================================
    # Calculate common values
    # ========================================================

    caring_words = [
        "okay",
        "safe",
        "eat",
        "sleep",
        "take care",
        "are you",
        "please",
        "reminder",
        "drink water",
        "don't forget"
    ]

    funny_words = [
        "lol",
        "lmao",
        "haha",
        "rofl",
        "lmfao"
    ]

    for person in sorted(participants):

        person_messages = data[person]["messages"]
        count = data[person]["count"]

        if count == 0:
            continue

        # ----------------------------------------------------
        # SPAMMER
        # Average consecutive message burst > 3
        # ----------------------------------------------------

        bursts = []
        current_burst = 0
        previous_sender = None

        for msg in messages:

            sender = msg["sender"]

            if sender == person:

                if previous_sender == person:
                    current_burst += 1

                else:
                    current_burst = 1

                previous_sender = person

            else:

                if current_burst > 0:
                    bursts.append(current_burst)

                current_burst = 0
                previous_sender = sender

        if current_burst > 0:
            bursts.append(current_burst)

        if len(bursts) > 0:
            average_burst = sum(bursts) / len(bursts)
        else:
            average_burst = 0

        # ----------------------------------------------------
        # GROUP MOM
        # ----------------------------------------------------

        caring_count = 0

        for msg in person_messages:

            text = msg["message"].lower()

            for word in caring_words:

                if word in text:
                    caring_count += 1

        # ----------------------------------------------------
        # NIGHT OWL
        # 23:00 - 04:59
        # ----------------------------------------------------

        night_count = 0

        for msg in person_messages:

            hour = msg["timestamp"].hour

            if hour >= 23 or hour <= 4:
                night_count += 1

        night_percentage = (
            night_count / count
        ) * 100

        # ----------------------------------------------------
        # STORYTELLER
        # ----------------------------------------------------

        total_words = 0

        for msg in person_messages:

            words = msg["message"].split()

            total_words += len(words)

        average_words = total_words / count

        # ----------------------------------------------------
        # DRAMA QUEEN
        # ----------------------------------------------------

        all_caps_count = 0
        exclamation_count = 0

        for msg in person_messages:

            text = msg["message"].strip()

            if len(text) >= 3:

                if text.isupper() and any(
                    char.isalpha() for char in text
                ):
                    all_caps_count += 1

            if "!!" in text:
                exclamation_count += 1

        all_caps_percentage = (
            all_caps_count / count
        ) * 100

        # ----------------------------------------------------
        # GHOST
        # ----------------------------------------------------

        dates = set()

        for msg in person_messages:
            dates.add(
                msg["timestamp"].date()
            )

        total_chat_days = (
            max(
                msg["timestamp"].date()
                for msg in messages
            )
            -
            min(
                msg["timestamp"].date()
                for msg in messages
            )
        ).days + 1

        silent_percentage = (
            (total_chat_days - len(dates))
            / total_chat_days
        ) * 100

        # ----------------------------------------------------
        # COMEDIAN
        # ----------------------------------------------------

        funny_count = 0

        for msg in person_messages:

            text = msg["message"].lower()

            for funny_word in funny_words:

                if funny_word in text:
                    funny_count += 1

        funny_percentage = (
            funny_count / count
        ) * 100

        # ----------------------------------------------------
        # QUESTION MASTER
        # ----------------------------------------------------

        question_count = 0

        for msg in person_messages:

            if msg["message"].strip().endswith("?"):
                question_count += 1

        question_percentage = (
            question_count / count
        ) * 100

        # ====================================================
        # SCORE / RULES
        # ====================================================

        scores = {}

        # SPAMMER
        if average_burst > 3:
            scores["SPAMMER"] = average_burst

        # GROUP MOM
        scores["GROUP MOM"] = caring_count

        # NIGHT OWL
        if night_percentage > 60:
            scores["NIGHT OWL"] = night_percentage

        # STORYTELLER
        if average_words > 30:
            scores["STORYTELLER"] = average_words

        # DRAMA QUEEN
        if (
            all_caps_percentage > 30
            or exclamation_count >= 2
        ):
            scores["DRAMA QUEEN"] = (
                all_caps_percentage
                + exclamation_count
            )

        # GHOST
        if silent_percentage > 60:
            scores["GHOST"] = silent_percentage

        # COMEDIAN
        scores["COMEDIAN"] = funny_percentage

        # QUESTION MASTER
        if question_percentage > 25:
            scores["QUESTION MASTER"] = question_percentage

        # ----------------------------------------------------
        # Select highest scoring archetype
        # ----------------------------------------------------

        if len(scores) == 0:

            archetype = "UNCLASSIFIED"

        else:

            best_score = max(scores.values())

            possible = []

            for name in archetype_order:

                if name in scores:
                    if scores[name] == best_score:
                        possible.append(name)

            if len(possible) > 0:
                archetype = possible[0]
            else:
                archetype = "UNCLASSIFIED"

        results[person] = {
            "archetype": archetype,
            "average_burst": average_burst,
            "caring_count": caring_count,
            "night_percentage": night_percentage,
            "average_words": average_words,
            "all_caps_percentage": all_caps_percentage,
            "exclamation_count": exclamation_count,
            "silent_percentage": silent_percentage,
            "funny_percentage": funny_percentage,
            "question_percentage": question_percentage
        }

    # ========================================================
    # DISPLAY ARCHETYPES
    # ========================================================

    print("\n" + "=" * 65)
    print("PERSONALITY ARCHETYPES")
    print("=" * 65)

    for person in sorted(results):

        result = results[person]

        print(
            f"{person:<15} -> "
            f"{result['archetype']}"
        )

    return results


# ============================================================
# FEATURE 8: FINAL FORMATTED REPORT
# ============================================================

def final_report(
    messages,
    participants,
    results
):

    if len(messages) == 0:
        return

    dates = [
        msg["timestamp"].date()
        for msg in messages
    ]

    start_date = min(dates)
    end_date = max(dates)

    total_days = (
        end_date - start_date
    ).days + 1

    person_counts = {}

    for person in participants:
        person_counts[person] = 0

    for msg in messages:
        person_counts[msg["sender"]] += 1

    most_active_person = max(
        person_counts,
        key=person_counts.get
    )

    day_counts = {}

    for msg in messages:

        date_value = msg["timestamp"].date()

        if date_value not in day_counts:
            day_counts[date_value] = 0

        day_counts[date_value] += 1

    busiest_day = max(
        day_counts,
        key=day_counts.get
    )

    hour_counts = {}

    for msg in messages:

        hour = msg["timestamp"].hour

        if hour not in hour_counts:
            hour_counts[hour] = 0

        hour_counts[hour] += 1

    busiest_hour = max(
        hour_counts,
        key=hour_counts.get
    )

    print("\n")
    print("╔" + "═" * 63 + "╗")
    print("║" + " GROUPDNA - WHATSAPP CHAT ANALYSIS ".center(63) + "║")
    print("╠" + "═" * 63 + "╣")

    print(
        "║ "
        f"Total Messages : {len(messages)}"
        .ljust(62)
        + "║"
    )

    print(
        "║ "
        f"Participants   : {len(participants)}"
        .ljust(62)
        + "║"
    )

    print(
        "║ "
        f"Date Range     : {start_date} to {end_date}"
        .ljust(62)
        + "║"
    )

    print(
        "║ "
        f"Total Days     : {total_days}"
        .ljust(62)
        + "║"
    )

    print("╠" + "═" * 63 + "╣")

    print(
        "║ "
        f"Most Active    : {most_active_person}"
        .ljust(62)
        + "║"
    )

    print(
        "║ "
        f"Busiest Day    : {busiest_day}"
        .ljust(62)
        + "║"
    )

    print(
        "║ "
        f"Busiest Hour   : {busiest_hour:02d}:00"
        .ljust(62)
        + "║"
    )

    print("╠" + "═" * 63 + "╣")

    print(
        "║ "
        + "PERSONALITY ARCHETYPES"
        .ljust(62)
        + "║"
    )

    print("╠" + "─" * 63 + "╣")

    for person in sorted(results):

        archetype = results[person]["archetype"]

        line = f"{person:<15} : {archetype}"

        print(
            "║ "
            + line.ljust(61)
            + "║"
        )

    print("╠" + "═" * 63 + "╣")

    print(
        "║ "
        + "PROJECT CONCEPTS USED"
        .ljust(62)
        + "║"
    )

    print("╠" + "─" * 63 + "╣")

    concepts = [
        "Strings",
        "Lists",
        "Dictionaries",
        "Sets",
        "Tuples",
        "Loops",
        "Conditionals",
        "Functions",
        "NumPy",
        "File I/O"
    ]

    for concept in concepts:

        print(
            "║ "
            + ("✓ " + concept).ljust(61)
            + "║"
        )

    print("╚" + "═" * 63 + "╝")


# ============================================================
# MAIN PROGRAM
# ============================================================

def main():

    print("\n")
    print("=" * 65)
    print("       GROUPDNA - WHATSAPP GROUP CHAT ANALYZER")
    print("=" * 65)

    # --------------------------------------------------------
    # FEATURE 1
    # --------------------------------------------------------

    try:

        (
            messages,
            participants,
            system_messages,
            deleted_messages,
            media_messages
        ) = parse_chat(FILE_NAME)

    except FileNotFoundError:

        print("\nERROR: Dataset file not found.")
        print(
            f"Please place '{FILE_NAME}' "
            "in the same folder as this Python file."
        )

        return

    # --------------------------------------------------------
    # Parser information
    # --------------------------------------------------------

    print("\nPARSER SUMMARY")
    print("-" * 65)

    print("Parsed Messages :", len(messages))
    print("Participants    :", len(participants))
    print("System Messages :", system_messages)
    print("Deleted Messages:", deleted_messages)
    print("Media Messages  :", media_messages)

    print("\nParticipants:")

    for person in sorted(participants):
        print("-", person)

    # --------------------------------------------------------
    # FEATURE 2
    # --------------------------------------------------------

    group_overview(
        messages,
        participants
    )

    # --------------------------------------------------------
    # FEATURE 3
    # --------------------------------------------------------

    busiest_day_and_hour(
        messages
    )

    # --------------------------------------------------------
    # FEATURE 4
    # --------------------------------------------------------

    activity_heatmap(
        messages,
        participants
    )

    # --------------------------------------------------------
    # FEATURE 5
    # --------------------------------------------------------

    top_words(
        messages
    )

    # --------------------------------------------------------
    # FEATURE 6
    # --------------------------------------------------------

    response_speed(
        messages
    )

    silent_streaks(
        messages,
        participants
    )

    # --------------------------------------------------------
    # FEATURE 7
    # --------------------------------------------------------

    archetype_results = calculate_archetypes(
        messages,
        participants
    )

    # --------------------------------------------------------
    # FEATURE 8
    # --------------------------------------------------------

    final_report(
        messages,
        participants,
        archetype_results
    )

    # --------------------------------------------------------
    # END
    # --------------------------------------------------------

    print("\nAnalysis completed successfully!")
    print("GroupDNA project finished.")


# ============================================================
# RUN PROGRAM
# ============================================================

if __name__ == "__main__":
    main()



       GROUPDNA - WHATSAPP GROUP CHAT ANALYZER

PARSER SUMMARY
-----------------------------------------------------------------
Parsed Messages : 0
Participants    : 0
System Messages : 3178
Deleted Messages: 0
Media Messages  : 0

Participants:
No messages found.

MOST ACTIVE DAY AND HOUR
No messages available to determine busiest day and hour.

ACTIVITY HEATMAP
No messages or participants available to generate activity heatmap.
------------------------------------------------------------------------------------------

TOP 10 COMMON WORDS

PERSONALITY ARCHETYPES

Analysis completed successfully!
GroupDNA project finished.
